# Project 1: Data Cleaning
Submitted on October 12, 2025

NLP1000 S17
Group 12: 
| Member Name      | ID Number      |
| ------------- | ------------- |
| Alcantara, Van Asher | 12340898 |
| Aragon, Enrique | 12227811 |
| Clavano, Angelica (Jack) | 12206245 |
| Lozada, Job | 12307246 |

## Table of Contents
1. Data Selection & Web Scraping
2. Data Cleaning and Segmentation
3. Parallel Corpus

---

## Running this Notebook:
- Make sure you have at least Python 3.12.0 or 3.14.0 installed.
- See first code block under 1.1 for `pip requirements`.

---

## Folder Structure:
```
/nlp1000 --> root
└── /data
    └── lang.txt files --> raw webscraped data
    └── /cleaned
        └── (sentences)-lang-cleaned.txt --> cleaned sentence files in txt format
        └── lang-cleaned.txt --> cleaned verse files in txt format
└── /parallel-corpora --> parallel corpora (aligned) but in separate xlsx files
└── .gitignore
└── main.ipynb --> MAIN SUBMISSION contains source information, source code, documentation
└── cleaned_sentences.ipynb --> cleaned sentence files in xlsx format - from data/cleaned/(sentences)-(lang)-cleaned.txt files
└── README.md --> contains the same information as this markdown block.
└── cleaned_verses.xlsx --> cleaned verse files in xlsx format - created when running the notebook from data/cleaned/(lang)-cleaned.txt
└── parallel_corpora.xlsx --> parallel corpora base file (unaligned) - created when running the notebook 
└── parallel_corpora_all.xlsx --> parallel corpora (aligned)
└── ai_declaration.pdf --> AI declaration
└── steps.xlsx --> steps for regex present in divide_into_verses() and divide_into_sentences()
```

## 1. Data Selection

For all 16 languages, we used the books of the Gospel, which are Matthew, Mark, Luke, and John, as our corpora. We initially considered using the books of Genesis and Exodus as the corpora, but the translations for more niche languages typically only spanned the New Testament. All corpora were sourced from https://www.bible.com/. Later, a word count summary will be provided.

| Target Language      | Link to Source      | Comments
| ------------- | ------------- |  ------------- | 
| Spanish  | https://www.bible.com/versions/1076-jbs-biblia-del-jubileo | |
| Tagalog | https://www.bible.com/versions/177-tlab-ang-biblia | |
| English | https://www.bible.com/versions/3523-nrsvue-new-revised-standard-version-updated-edition-2021 | |
| Hiligaynon/Ilonggo | https://www.bible.com/versions/2190-mbbhil12-maayong-balita-nga-biblia-2012 | Listed as Ilonggo on the website |
| Bikol/Bikolano | https://www.bible.com/versions/890-mbbbik92-marahay-na-bareta-biblia |  |
| Waray | https://www.bible.com/versions/2198-mbbsam-samarenyo-meaning-based-bible-1984 | |
| Ilocano | https://www.bible.com/versions/782-ripv-ti-baro-a-naimbag-a-damag-biblia | |
| Cebuano | https://www.bible.com/versions/562-rcpv-ang-bag-ong-maayong-balita-biblia | |
| Kapampangan | https://www.bible.com/versions/1141-pmpv-ing-mayap-a-balita-biblia | |
| Pangasinense | https://www.bible.com/versions/2194-mbbpan83-maung-a-balita-biblia | |
| Yakan | https://www.bible.com/versions/1388-yakv-yakan | |
| Ivatan | https://www.bible.com/versions/1315-vtsp-ivatan | |
| Tausug  | https://www.bible.com/versions/1319-tsg-kitab-injil | |
| Yami  | https://www.bible.com/versions/2364-snt-seysyo-no-tao | |
| Tuwali Ifugao | https://www.bible.com/versions/2123-ifkwb-nan-kalin-apu-dios | Listed as "tuwali_ifugao" in project files |
| Masbateño/Masbatenyo | https://www.bible.com/versions/1222-msb-masbatenyo | Listed as "masbateno" in project files |
| TOTAL | 16 | |

### 1.1 Web Scraping

Using `webscraper.py`, we were able to extract all the Gospels in the languages listed in Section 1. First, the bible abbreviation and number is compiled, then for each of the chapters and their chapter ranges (mat, mrk, luk, jhn) -- which are all listed the same on Bible.com -- it will scrape for the text and compile them in the `/data` folder. This may take around 3-5 min. If on public wifi, expect around 25-40 minutes.

The raw data has been saved ahead with the `/data` folder under the following naming convention: `lang.txt`

In [ ]:
from bs4 import BeautifulSoup       # pip install beautifulsoup4
from pathlib import Path            # pip install pathlib
import pandas as pd                 # pip install pandas
                                    # pip install xlsxwriter
                                    # pip install ipykernel
                                    # pip install openpyxl 
                                    
from urllib.request import urlopen 
import re

Using information from the URLs of the Gospels, we compile all of them here. Note that all bibles hosted on Bible.com will have the same book codes.

In [ ]:
languages = ["spanish", "tagalog", "english", "hiligaynon", "bikol", "waray", "ilocano", "cebuano", "kapampangan", "pangasinense", "yakan", "ivatan", "tausug", "yami", "tuwali_ifugao", "masbateno"]
bibleNumbers = ["1076", "177", "3523", "2190", "890", "2198", "782", "562", "1141", "2194", "1388", "1315", "1319", "2364", "2123", "1222"]
bibleAbbreviation = ["JBS", "TLAB", "NRSVUE", "MBBHIL12", "MBBBIK92", "MBBSAM", "RIPV", "RCPV", "PMPV", "MBBPAN83", "YAKV", "VTSP", "TSG", "SNT", "IFKWB", "MSB"]

bookCodes = ["mat", "mrk", "luk", "jhn"]
chapterRanges = {
    "mat": [1, 28],
    "mrk": [1, 16],
    "luk": [1, 24],
    "jhn": [1, 21]
}

Next, we began to webscrape using the following code. For all languages listed above, associated with their bible number and abbreviation, it will proceed to scrape every chapter of Matthew, Mark, Luke, and John.

All data from this webscraper is located in `/data` with the naming convention `lang.txt`

In [ ]:
# this has been commented out so that it won't run every time. feel free to delete the data folder if you would like to rescrape.
# latest scrape: October 12, 2025
'''
# webscraper v4; adjusted to use 'Path' instead of 'os'
# instead of "!python webscraper_v3.py"
output_folder = "data"
data_folder = Path("data")
data_folder.mkdir(parents=True, exist_ok=True) 

# clear content of the file if it exists so it doesn't make copies
for lang in languages:
    file_path = data_folder / f"{lang}.txt"
    file_path.write_text("", encoding="utf-8")

for lang, bibleNumber, abbreviation in zip(languages, bibleNumbers, bibleAbbreviation):
  # if in the previous code block, languages, bibleNumbers, and/or bibleAbbreviation is "SKIP", just skip it
  if bibleNumber == "SKIP" or abbreviation == "SKIP":
        continue
  
  for bookCode, (start, end) in chapterRanges.items():
     for chapter in range(start, end + 1):
      urlPartOne = "https://www.bible.com/bible/"
      chapterNumber = str(chapter)
      bibleName = f".{abbreviation}"

      url = f"{urlPartOne}{bibleNumber}/{bookCode.upper()}.{chapterNumber}{bibleName}"
      print(f"Scraping URL: {url}")

      try: 
        page = urlopen(url)
        html = page.read().decode("utf-8")
        soup = BeautifulSoup(html, "html.parser")
        
        #IMPORTANT: THIS ONLY WORKS ON BIBLE.COM, for other sites just use inspect element then select the div/class containing the text so that it extracts just that
        text = soup.find('div', {'class': 'ChapterContent_reader__Dt27r'})

        if text:
          # excludes the pop-up notes
          for note in text.select('span.ChapterContent_note__YlDW0'):
            note.decompose()
          
          # excludes chapter headings
          for heading in text.select('span.ChapterContent_heading__xBDcs'):
            heading.decompose()
        
          # write to lang-specific file
          file_path = Path(output_folder) / f"{lang}.txt" # file_name = os.path.join(output_folder, f"{lang}.txt")

          with file_path.open("a", encoding="utf-8") as f: # with open(file_name, "a", encoding="utf-8") as f:
              f.write(text.get_text() + "\n") 
        else:
            print(f"No content found for {lang}, {bookCode}, chapter {chapter}")
      except Exception as e:
                  print(f"Error scraping {url}: {e}")'''

The text data as is will show up in the following format: 

```txt
(book)(space)(book number)
(new line)
(chapter)(title)(subtitle if any)(space)(line number)(content)
```

where the `(line number)(content)` repeats for the entire page, and then a new line is made for the next chapter to begin. This can be cleaned using Regular Expressions/regex.


## 2. Data Cleaning & Segmentation

The following function, `divide_into_verses` takes the raw lang.txt files and divides them into verses. The steps can be seen on the Excel sheet attached called `steps.xlsx`.

In [3]:
# changed all \t to |
def divide_into_verses(text):
    # WEIRD ENCODING: replace weird ‘ with ' accounts for letters before and after (Jack)
    text = re.sub(r"(\w)‘(\w)", r"\1'\2", text, flags=re.MULTILINE)
    text = re.sub(r"‘", r"'", text, flags=re.MULTILINE)  # for standalone 

    # WEIRD ENCODING: replace ’ with ' (Jack)
    text = re.sub(r"(\w)’(\w)", r"\1'\2", text, flags=re.MULTILINE)
    text = re.sub(r"’", r"'", text, flags=re.MULTILINE)  # for standalone 
    
    # WEIRD ENCODING: replace weird ” and “ with " and accounts for letters before and after (Jack)
    text = re.sub(r"(\w)”", r'\1"', text, flags=re.MULTILINE)
    text = re.sub(r"“(\w)", r'"\1', text, flags=re.MULTILINE)
    text = re.sub(r"”", r'"', text, flags=re.MULTILINE)  # for standalone
    text = re.sub(r"“", r'"', text, flags=re.MULTILINE)  # for standalone

    # SPACING: remove double spaces
    text = re.sub(r' {2,}', ' ', text, flags=re.MULTILINE)

    # SPACING: remove leading newlines
    text = re.sub(r'^\s*$', r'', text, flags = re.MULTILINE)

    # SPACING: remove leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    #BASIC SPLITTING
    text = re.sub(r'^\d+ (\d+)', r'\1', text, flags = re.MULTILINE)
    text = re.sub(r'(\.\s)(\d+\s)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\.\s)(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    
    text = re.sub(r'(\s)(\d+)([A-z])', r'\1\n\2\3', text, flags = re.MULTILINE)
    #for verses next to quotes
    text = re.sub(r'(\s)(\d+\")', r'\1\n\2', text, flags = re.MULTILINE)
    #next to single quotes to the right
    text = re.sub(r'\s(\d+\s\')', r'\n\1', text, flags = re.MULTILINE)
    #double quote before the new verse
    text = re.sub(r'(\")\s(\d+\s)', r'\1\n\2', text, flags = re.MULTILINE)
    
    #colon before new verse
    text = re.sub(r'(:)\s(\d+\s)', r'\1\n\2', text, flags = re.MULTILINE)
    #comma before new verse
    text = re.sub(r'(,)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)
    # ; before new verse (Jack)
    text = re.sub(r'(;)\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    # ) before new verse (Jack)
    text = re.sub(r'(\))\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    # ] before new verse (Jack)
    text = re.sub(r'(\])\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    #! before new verse
    text = re.sub(r'(!)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)
    #? before new verse
    text = re.sub(r'(\?)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)
    #— before new verse
    text = re.sub(r'(—)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)

    # add space between number and capital letter (Jack)
    text = re.sub(r'(\d+)([A-Z])', r'\1 \2', text, flags=re.MULTILINE)
    
    #adding tabs for splitting
    text = re.sub(r'(\d+-\d+)', r'\n\1', text, flags=re.MULTILINE)
    text = re.sub(r'(\d+[A-z]-\d+)', r'\1', text, flags = re.MULTILINE)
    
    # handling 12,13 in tuwali-ifugao
    text = re.sub(r'(\d+,\d+)', r'\1|', text, flags = re.MULTILINE)

    #split verse ranges from text first
    text = re.sub(r'^(\d+-\d+)', r'\1|', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+[A-z]-\d+)', r'\1|', text, flags = re.MULTILINE)

    # ( before new verse (Jack)
    text = re.sub(r'(\d+)[\)\(]', r'\1|', text, flags=re.MULTILINE)
    # " before new verse (Jack)
    text = re.sub(r'(\")\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    text = re.sub(r'\s(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    # [ before new verse (Jack)
    text = re.sub(r'(\[)\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\[)(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    
    #splitting the rest of it with tabs
    text = re.sub(r'^(\d+)([A-z][^-])', r'\1|\2', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+)\s', r'\1|', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+)\s(\")', r'\1|\2', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+)([\"\'])', r'\1|\2', text, flags = re.MULTILINE)
    # for Matthew|7|24 "Everyone... in english 
    text = re.sub(r'^(\d+)\s(")', r'\1|\2', text, flags=re.MULTILINE) # (Jack, Enrique)
    
    # remove empty lines
    text = re.sub(r'^\n', r'', text, flags=re.MULTILINE)
    # remove lines that only contain [ (Jack)
    text = re.sub(r'(\d+\|)(\[)', r'\1', text, flags=re.MULTILINE)
    # remove lines that have nothing after |juan|8|8| ******
    text = re.sub(r'^(\w+\|\d+\|\w+\|\s*$)', r'', text, flags=re.MULTILINE)
    
    # ----- OTHER SPECIFIC QUIRKS BELOW -----
    # spanish
    text = re.sub(r'¶\s', r'|', text, flags = re.MULTILINE) # (Enrique)
    text = re.sub(r'¶', r'|', text, flags = re.MULTILINE) # (Jack)

    # handling upside down ? and !  (Enrique)
    text = re.sub(r'^(\d+)(\()', r'\1|\2', text, flags=re.MULTILINE)
    text = re.sub(r'^(\d+)(\¿)', r'\1|\2', text, flags = re.MULTILINE) 
    text = re.sub(r'^(\d+)(\¡)', r'\1|\2', text, flags = re.MULTILINE) 
    text = re.sub(r'^(\d+)(\?)', r'\1|\2', text, flags = re.MULTILINE) 

    # handling (39An in bikol and Matay|4|40) in yami 
    text = re.sub(r'(\()(\d+[A-z ])', r'\1\n\2', text, flags = re.MULTILINE) # text = re.sub(r'(\()(\d+[^\)])', r'\1\n \2', text, flags = re.MULTILINE) **********
    text = re.sub(r'(\[)(\d+[^\]])', r'\1\n\2', text, flags=re.MULTILINE) # and [9Pagkabuhay in bikol
    text = re.sub(r'(\' )(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # and 7'For in english and ' 24 "Everyone, in english; removed space after \2 (Jack, Enrique)
    text = re.sub(r'(\" )(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # for 29|(And all the people in english (Jack, Enrique)
    text = re.sub(r'^(\d+)(\')', r'\1|\2', text, flags=re.MULTILINE) # for 26'I in english and 6'Sinasabihan in bikol (Jack, Enrique)
    # for John|8|11| ... again."]] 12 Again Jesus ... in english (Jack, Enrique)
    text = re.sub(r'(\]\]|\\"|\')\s+(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    # for Luke|2|22| ... Lord 23|(as it ... (Jack, Enrique)
    text = re.sub(r'(\d+)\|\(', r'\n\1 (', text, flags=re.MULTILINE)
    # for Luke|2|23 (as in english (Jack, Enrique)
    text = re.sub(r'^(\d+)\s(\()', r'\1|\2', text, flags=re.MULTILINE) # ****** ADDED ^

    # remove lines that have nothing after it or whitespace |juan|8|8| AGAIN ******
    text = re.sub(r'^(\w+\|\d+\|\w+\|\s*$)', r'', text, flags=re.MULTILINE)

    # remove 000| if not at the start of the line ********** 
    text = re.sub(r'000\|', r'000', text, flags=re.MULTILINE)

    # remove (number| ********** for yami
    text = re.sub(r'\((\d+)\|', r'\1', text, flags=re.MULTILINE) # text = re.sub(r'^(\w+\|\d+\|\d+\|.*)(\(\d+\|)', r'\1\2', text, flags=re.MULTILINE)

    # remove pipe in 3|B but after the first few pipes ********** (\w+\|\d+\|\d+\|.*)(\d+)\|([A-Z]) for yami **********
    text = re.sub(r'(\w+\|\d+\|\d+\|.*)(\d+)\|([A-Z].*?)', r'\1\2\3', text, flags=re.MULTILINE)
    text = re.sub(r'(\w+\|\d+\|\d+\|.*)(\d+)\|([A-Z].*?)', r'\1\2\3', text, flags=re.MULTILINE) # lowercase ver

    # add new line to make sure 44| and 55| are accounted for (Jack)
    text = re.sub(r'\s(\d+\|)', r'\n\1', text)

    # for Luke|2|23 (as in english but for cebuano this time AGAIN **********
    text = re.sub(r'^(\d+)\s(\()', r'\1|\2', text, flags=re.MULTILINE) # ****** ADDED ^
    # for Matthew|7|24 " in english but for cebuano this time AGAIN **********
    text = re.sub(r'^(\d+)\s(\")', r'\1|\2', text, flags=re.MULTILINE) # ****** 

    # for 10,11|ot in tuwali_ifugao (Jack)
    # reference text = re.sub(r'(\d+,\d+)', r'\1|', text, flags = re.MULTILINE)
    text = re.sub(r'(\d+,\d+\|)', r'\n\1', text, flags = re.MULTILINE)

    # " before new verse AGAIN (Jack)
    text = re.sub(r'(\")\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    text = re.sub(r'\s(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space

    # formatting for csv/xlsx, delimiter is | or pipe (Enrique)
    finalText = []
    bookCh = ""
    
    for line in text.splitlines():
        if line:
            if line[0] not in ["0","1","2","3","4","5","6","7","8","9"]:
                bookCh = line.strip()
            else:
                line = bookCh + "|" +line
        finalText.append(line)


    text= '\n'.join(finalText)

    text = re.sub(r'^([SMJLY].*)\d+(\n)', r'', text, flags = re.MULTILINE)

    text = re.sub(r'^([SMJLY][^|]*?)\s+(\d+)', r'\1|\2', text, flags=re.MULTILINE)
    
    text = "Book|Chapter #|Verse #|Verse\n" + text

    # remove lines that have nothing after it or whitespace |juan|8|8| AGAIN (Jack)
    text = re.sub(r'^(\w+\|\d+\|\w+\|\s*$)', r'', text, flags=re.MULTILINE)

    # remove empty lines AGAIN (Jack)
    text = re.sub(r'^\n', r'', text, flags=re.MULTILINE)

    return text

# i forgot who did what na correct me if i'm wrong

The following function, `divide_into_sentences` takes the raw lang.txt files and divides them into verses. The steps can be seen on the Excel sheet attached called `steps.xlsx`

In [ ]:
def divide_into_sentences(text):
    # remove any numbers with spaces at the start of lines
    # example: "1 This is a verse -> "This is a verse"
    text = re.sub(r'^\d+\s*', r'', text, flags=re.MULTILINE)
    
    # removes book and chapter headings (e.g., "Matthew 1")
    # ^[A-Za-zÀ-ÖØ-öø-ÿ\s]+ matches the book name (also covers accented letters for other languaes that do have them)
    # \s+ matches the space between the book name and chapter number
    # \d+ matches the chapter number
    # \s*\n? matches any trailing spaces and newline
    text = re.sub(r'^[0-9A-Za-zÀ-ÖØ-öø-ÿ\s]+\d+\s*\n?', '', text, flags=re.MULTILINE)

    # removes verse numbers
    # (?![\d,\.]) makes sure we don't remove numbers that are part of decimals or commas
    text = re.sub(r'\b\d+(?![\d,\.])', '', text)

    # replaces multiple newlines with a single newline
    text = re.sub(r'\n+', r'\n', text)

    # replaces multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # ensures there is a space after sentence-ending punctuation if followed by a non-space character
    # [.!?] matches sentence ending punctuation
    # ["”’\'\)\]\}] matches any closing quotes, parentheses, or brackets that may follow the punctuation
    # (?=\S) ensures the punctuation is followed by a non-space character
    text = re.sub(r'([.!?]["”’\'\)\]\}]?)(?=\S)', r'\1 ', text)

    # remove spaces after opening punctuation
    text = re.sub(r'([“‘\(\[])\s+', r'\1', text)

    # remove spaces before closing punctuation
    text = re.sub(r'\s+([”’\)\]])', r'\1', text)

    # ensures each sentence starts on a new line
    # [.!?] matches sentence ending punctuation
    # ["\'”’\)\]\}] matches any closing quotes, parentheses, or brackets that may follow the punctuation
    # \s+ matches the whitespace following the punctuation
    text = re.sub(r'([.!?](?:["\'”’\)\]\}]+)?)\s+', r'\1\n', text)

    # This regex keeps letters (including accented), numbers, and common punctuation marks 
    # while removing inconsistent special characters.
    # Basically, if its not in the list of characters, it gets omitted.
    # I had to consult AI with this one in giving me the characters to be included.
    text = re.sub(r'[^A-Za-zÀ-ÖØ-öø-ÿ\u00C0-\u024F\u1E00-\u1EFF0-9\s\.\,\!\?\:\;\'"“”‘’\-\(\)]', "", text)
    
    # removes leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    # brings closing parentheses to the previous line if they are on a new line
    text = re.sub(r'\n\)', r')\n', text)

    # removes white space at the start of each line after ", ', or )"
    text = re.sub(r'^([\"\'\)])\s+', r'\1', text, flags=re.MULTILINE)

    # removes leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    # standardizes quotes to " and ' at the end
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[‘’]', "'", text)

    return text

NOTE: Please delete the cleaned folder before clicking `Run All`. The code below prepares the data/cleaned folder.

In [ ]:
# data_folder = Path("data") already exists above
cleaned_folder = Path("data/cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True) 

We will run `divide_into_verses` on the raw webscraping .txts in `/data`. This will be used for the Excel sheets later in the project.

In [ ]:
for lang in languages:
    data_folder = Path("data")
    file_path = data_folder / f"{lang}.txt"
    output_path = cleaned_folder / f"{lang}-cleaned.txt"

    # SKIP if the input file does not exist or is empty
    if not file_path.exists() or file_path.stat().st_size == 0:
        print(f"Skipped: {file_path} (file does not exist or is empty)")
        continue

    # DELETE IF EXISTS
    if output_path.exists():
        print(f"unlinking/deleting old version of {lang}-cleaned.txt")
        output_path.unlink()

    # read input
    with file_path.open("r", errors="ignore", encoding="utf-8") as f:
        text = f.read()

    # clean the text
    cleaned_text = divide_into_verses(text)

    # save
    with output_path.open("w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned and saved: {output_path}")

Next, we will run `divide_into_sentences` on the raw data.

In [ ]:
for lang in languages:
    data_folder = Path("data")
    file_path = data_folder / f"{lang}.txt"
    output_path = cleaned_folder / f"(sentences)-{lang}-cleaned.txt"

    # SKIP if the input file does not exist or is empty
    if not file_path.exists() or file_path.stat().st_size == 0:
        print(f"Skipped: {file_path} (file does not exist or is empty)")
        continue

    # DELETE IF EXISTS
    if output_path.exists():
        print(f"unlinking/deleting old version of {lang}-cleaned.txt")
        output_path.unlink()

    # read input
    with file_path.open("r", errors="ignore", encoding="utf-8") as f:
        text = f.read()

    # clean the text
    cleaned_text = divide_into_sentences(text)

    # save
    with output_path.open("w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned and saved: {output_path}")

Here, we will be getting the word count of the raw `lang.txt` files, freshly scraped from Bible.com, and then a word count of the `(sentences)-lang.txt` files.

In [ ]:
data_folder = Path("data")
total_word_count = 0

print("PRE-CLEAN WORD COUNT")

for lang in languages:
    cleaned_file_path = data_folder / f"{lang}.txt"  

    if cleaned_file_path.exists() and cleaned_file_path.stat().st_size > 0:
        with cleaned_file_path.open("r", encoding="utf-8") as f:
            text = f.read()
            word_count = len(text.split())
            total_word_count += word_count
            print(f"Word count for {lang}.txt: {word_count}")
    else:
        print(f"Skipped: {cleaned_file_path} (file does not exist or is empty)")

print(f"Total word count across all -cleaned files: {total_word_count}")

In [ ]:
data_folder = Path("data/cleaned")
total_word_count = 0

print("SENTENCE WORD COUNT")

for lang in languages:
    sentence_file_path = data_folder / f"(sentences)-{lang}-cleaned.txt"  

    if sentence_file_path.exists() and sentence_file_path.stat().st_size > 0:
        with sentence_file_path.open("r", encoding="utf-8") as f:
            text = f.read()
            word_count = len(text.split())
            total_word_count += word_count
            print(f"Word count for {lang}-cleaned.txt: {word_count}")
    else:
        print(f"Skipped: {sentence_file_path} (file does not exist or is empty)")

print(f"Total word count across all (sentences)-cleaned files: {total_word_count}")

SUMMARY:
| Target Language      | Link to Source      | # Words Before Cleaning | # Words After Cleaning |
| ------------- | ------------- | ------------- |  ------------- |
| Spanish  | https://www.bible.com/versions/1076-jbs-biblia-del-jubileo | 76085 | 75318 |
| Tagalog | https://www.bible.com/versions/177-tlab-ang-biblia | 86888 |86596  |
| English | https://www.bible.com/versions/3523-nrsvue-new-revised-standard-version-updated-edition-2021 |  81210 | 80426 |
| Hiligaynon/Ilonggo | https://www.bible.com/versions/2190-mbbhil12-maayong-balita-nga-biblia-2012 | 92970 | 92876 |
| Bikol/Bikolano | https://www.bible.com/versions/890-mbbbik92-marahay-na-bareta-biblia | 79624 | 79543 |
| Waray | https://www.bible.com/versions/2198-mbbsam-samarenyo-meaning-based-bible-1984 | 89081 | 88941 |
| Ilocano | https://www.bible.com/versions/782-ripv-ti-baro-a-naimbag-a-damag-biblia | 73149 | 73030 |
| Cebuano | https://www.bible.com/versions/562-rcpv-ang-bag-ong-maayong-balita-biblia | 84973 | 84773 |
| Kapampangan | https://www.bible.com/versions/1141-pmpv-ing-mayap-a-balita-biblia | 84635 | 84254 |
| Pangasinense | https://www.bible.com/versions/2194-mbbpan83-maung-a-balita-biblia | 77748 | 77670 |
| Yakan | https://www.bible.com/versions/1388-yakv-yakan | 84123 | 83832 |
| Ivatan | https://www.bible.com/versions/1315-vtsp-ivatan | 97534 | 97250 |
| Tausug  | https://www.bible.com/versions/1319-tsg-kitab-injil | 113208 | 113228 |
| Yami  | https://www.bible.com/versions/2364-snt-seysyo-no-tao | 107967 | 107672 |
| Tuwali Ifugao | https://www.bible.com/versions/2123-ifkwb-nan-kalin-apu-dios | 82580 | 82321 |
| Masbateño/Masbatenyo | https://www.bible.com/versions/1222-msb-masbatenyo | 88302 | 88228 |
| TOTAL | 16 | 1400077 | 1395958 |

## 3. Parallel Corpus
For our implementation, we have selected the following languages: english(eng) tagalog(tgl) yami(tao) kapampangan(pam) pangasinense(pag)
- english-tagalog (ENG-TGL) - Jack
- english-yami (ENG-TAO) - Jack 
- tagalog-kapampangan (TGL-PAM) - Enrique
- tagalog-yami (TGL-TAO) - Jack
- english-pangasinense (ENG-PAG) - Enrique
- kapampangan-yami (PAM-TAO) - Enrique
- english-kapampangan (ENG-PAM) - Enrique

The code below creates the location for the `cleaned-verses` file. If it exists, unlink or delete the file so that it's completely overwritten each time this notebook is run and you are provided with a fresh file. Next, using pandas ExcelWriter, read_csv, and to_excel, load all the `(lang)-cleaned.txt` files into DataFrames. Then, using the df, export it to excel with each sheet named `(lang)-verses`. To remove the indexing of the DataFrame, `index=false`.

In [1]:
# CREATE CLEANED VERSES
# data_folder = Path("data/cleaned")
output_excel_path = Path("cleaned_verses.xlsx")
csv_folder = Path("data/csv")

# if cleaned_verses.xlsx exists just delete it
if output_excel_path.exists():
    print("unlinking/deleting old version of cleaned_verses.xlsx")
    output_excel_path.unlink()

languages = ["spanish", "tagalog", "english", "hiligaynon", "bikol", "waray", "ilocano", "cebuano", "kapampangan", "pangasinense", "yakan", "ivatan", "tausug", "yami", "tuwali_ifugao", "masbateno"]

# cleaned_verses
with pd.ExcelWriter(output_excel_path) as pd_writer:
    for lang in languages:
        print(f"attempting {lang}")
        cleaned_file_path = data_folder / f"{lang}-cleaned.txt"

        if cleaned_file_path.exists() and cleaned_file_path.stat().st_size > 0:
            try: 
                with cleaned_file_path.open("r", encoding="utf-8") as f:
                    df = pd.read_csv(cleaned_file_path, sep="\\|", on_bad_lines="warn", engine="python", quoting=3)
                    excel = df.to_excel(pd_writer, sheet_name=f'{lang}-verses', index=False, engine="python") # false indexing bc we already have it
                    print(f"Added {lang} to cleaned_verses.xlsx")
            except Exception as e:
                print(f"Error processing {lang}: {e}")

NameError: name 'Path' is not defined

Using the same approach, we used Python to create another file which will be used for the `parallel_corpora.xlsx`. This is the base file that we will be working with and only contains our target languages.

In [ ]:
# CREATE PARALLEL CORPORA BASE FILE
# data_folder = Path("data/cleaned")
output_excel_path = Path("parallel_corpora.xlsx")
csv_folder = Path("data/csv")

# if parallel_corpora.xlsx exists just delete it
if output_excel_path.exists():
    print("unlinking/deleting old version of parallel_corpora.xlsx")
    output_excel_path.unlink()

corpora_lang = ["english", "tagalog", "yami", "kapampangan", "pangasinense"]

# parallel corpora
with pd.ExcelWriter(output_excel_path) as pd_writer:
    for lang in corpora_lang:
        print(f"attempting {lang}")
        cleaned_file_path = data_folder / f"{lang}-cleaned.txt"

        if cleaned_file_path.exists() and cleaned_file_path.stat().st_size > 0:
            try: 
                with cleaned_file_path.open("r", encoding="utf-8") as f:
                    df = pd.read_csv(cleaned_file_path, sep="\\|", on_bad_lines="warn", engine="python", quoting=3)
                    excel = df.to_excel(pd_writer, sheet_name=f'{lang}-verses', index=False, engine="python") # false indexing bc we already have it
                    print(f"Added {lang} to parallel_corpora.xlsx")
            except Exception as e:
                print(f"Error processing {lang}: {e}")

print("Finished")

After some digging, we found out that there are some lines that are really missing on the website. Others also contain ranges like verse 9-10 are combined in one line for a specific language. In that regard, we will be manually cleaning the parallel corpora here onwards.

One of our methods were the following:
1. Using the base `parallel_corpora` file, make a new sheet or add a copy of one of the target languages.
2. Copy paste the "Book", "Chapter #", "Verse #", and "Verse" columns to the new sheet next to the target language.
3. Change the "Verse" header to the source and target language. 
4. Using `eng-tgl` (English/Tagalog) as an example: Use Excel's Conditional Highlighting to format all "Matthew" and "Mateo". Repeat for others like "Mark" and "Marcos". Not a cruicial step but helped me personally visually distinguish the chapters.
5. Scroll through the sheet, if the Verse column of language A does not match Langauge B, check if it's a missing verse in one of the languages or if it's because one of the languages has a range (ex. 9-10)
6. If it's a missing verse in Language A, move the entire row of Language A down, Copy the book, chapter number, and verse number onto Language A's table, and leave the verse blank.
7. If it's a range in Language B, if the range is X-Y, go to Language A's Y verse and copy it. Edit the X verse's Verse column to include "Y (rest of the verse here)" Then change the Verse Number of the merged column to X-Y. Delete the old Y row.
8. When all are cleaned and aligned, delete Language B's Book, Chapter #, and Verse #. Move the Verse Column (with "Verse" replaced as the target language name) next to the Verse column of Language A.

This method may work for a smaller dataset and two languages but it may be inefficient for larger datasets. Before resorting to manual cleaning/aligning, I turned all of the language sheets from the `parallel_corpora` base file into DataFrames and renamed all the book titles to match the target language. Then I used `pd.merge` to combine English and Tagalog. The plan was to use DataFrames and pandas for the entire process and then upload the resulting DataFrames into a sheet using `ExcelWriter`. This attempt fell short when I tried to handle the ranged rows, which proved to be more time consuming than just doing it manually. - Jack

In loving memory:
```python
eng_tgl = pd.merge(
    eng_df[['Book', 'Chapter #', 'Verse #', 'English']],
    tgl_df[['Book', 'Chapter #', 'Verse #', 'Tagalog']],
    on=['Book', 'Chapter #', 'Verse #'],
    how='outer'   # this keeps all verses even if missing in one language
)
```

## 4. AI Declaration

Please see the ai_declaration.pdf document in the same directory.

## Resources
---
Data from Bible.com
- Spanish: https://www.bible.com/versions/1076-jbs-biblia-del-jubileo
- Tagalog: https://www.bible.com/versions/177-tlab-ang-biblia
- English: https://www.bible.com/versions/3523-nrsvue-new-revised-standard-version-updated-edition-2021
- Hiligaynon/Ilonggo: https://www.bible.com/versions/2190-mbbhil12-maayong-balita-nga-biblia-2012
- Bikol/Bikolano: https://www.bible.com/versions/890-mbbbik92-marahay-na-bareta-biblia
- Waray: https://www.bible.com/versions/2198-mbbsam-samarenyo-meaning-based-bible-1984
- Ilocano: https://www.bible.com/versions/782-ripv-ti-baro-a-naimbag-a-damag-biblia
- Cebuano: https://www.bible.com/versions/562-rcpv-ang-bag-ong-maayong-balita-biblia
- Kapampangan: https://www.bible.com/versions/1141-pmpv-ing-mayap-a-balita-biblia
- Pangasinense: https://www.bible.com/versions/2194-mbbpan83-maung-a-balita-biblia
- Yakan: https://www.bible.com/versions/1388-yakv-yakan
- Ivatan: https://www.bible.com/versions/1315-vtsp-ivatan
- Tausug: https://www.bible.com/versions/1319-tsg-kitab-injil
- Yami: https://www.bible.com/versions/2364-snt-seysyo-no-tao
- Tuwali Ifugao: https://www.bible.com/versions/2123-ifkwb-nan-kalin-apu-dios
- Masbateño/Masbatenyo: https://www.bible.com/versions/1222-msb-masbatenyo

Libraries
- PathLib: https://docs.python.org/3/library/pathlib.html
- DataFrames: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html
- pd.merge: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html#pandas.DataFrame.merge
- pd.sort_values: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html
- pd.rename: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html

Sheet Handling
- https://stackoverflow.com/questions/42370977/how-to-save-a-new-sheet-in-an-existing-excel-file-using-pandas

Regex Testing: https://regex101.com/